In [1]:
!pip install -q openai python-dotenv pandas pillow


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import json
import base64
import pandas as pd
from PIL import Image, ImageDraw

from dotenv import load_dotenv
from openai import OpenAI

In [3]:
load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

print("Loaded:", bool(os.getenv("OPENAI_API_KEY")))

Loaded: True


In [4]:
DATASET_DIR = "dataset"
OUTPUT_DIR = "outputs_phase4"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [5]:
IMAGE_PATH = os.path.join(DATASET_DIR, "floorplan.jpg")   # change filename
IMAGE_PATH

'dataset\\floorplan.jpg'

In [6]:
img = Image.open(IMAGE_PATH)
img

FileNotFoundError: [Errno 2] No such file or directory: 'dataset\\floorplan.jpg'

In [ ]:
width, height = img.size
print("Width:", width)
print("Height:", height)

In [ ]:
def encode_image(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

In [ ]:
PROMPT = f"""
You are given a layout image of size width={width} and height={height} pixels.

Identify visible objects such as:
- sink
- stove
- refrigerator
- sofa
- table
- chair
- bed
- door
- window

Return estimated center pixel coordinates (x,y) for each detected object.

Return strict JSON only:

{{
  "objects": [
    {{
      "name": "",
      "x": 0,
      "y": 0
    }}
  ]
}}
"""

In [ ]:
def get_coordinates(path, model="gpt-5-nano"):
    
    img_b64 = encode_image(path)

    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": PROMPT},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{img_b64}",
                            "detail": "high"
                        }
                    }
                ]
            }
        ],
        max_tokens=1200
    )

    return response.choices[0].message.content

In [ ]:
raw_output = get_coordinates(IMAGE_PATH)
print(raw_output)

In [ ]:
import re

def extract_json(text):
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        return json.loads(match.group())
    return {"objects": []}

coords = extract_json(raw_output)
coords

In [ ]:
df = pd.DataFrame(coords["objects"])
df

In [ ]:
img_marked = img.copy()
draw = ImageDraw.Draw(img_marked)

for _, row in df.iterrows():
    x = int(row["x"])
    y = int(row["y"])
    name = row["name"]

    r = 8
    draw.ellipse((x-r, y-r, x+r, y+r), outline="red", width=3)
    draw.text((x+10, y-10), name, fill="red")

img_marked

In [ ]:
import math

errors = []

for _, row in df.iterrows():
    name = row["name"]

    if name in ground_truth:
        gx, gy = ground_truth[name]
        px, py = row["x"], row["y"]

        err = math.sqrt((gx-px)**2 + (gy-py)**2)

        errors.append({
            "object": name,
            "pred_x": px,
            "pred_y": py,
            "true_x": gx,
            "true_y": gy,
            "pixel_error": round(err,2)
        })

err_df = pd.DataFrame(errors)
err_df

In [ ]:
textract_output = {
    "Blocks": [
        {
            "BlockType": "WORD",
            "Text": "KITCHEN FLOOR PLAN",
            "Geometry": {
                "BoundingBox": {
                    "Left": 0.34,
                    "Top": 0.02,
                    "Width": 0.30,
                    "Height": 0.04
                }
            }
        },

        {
            "BlockType": "WORD",
            "Text": "REFRIGERATOR",
            "Geometry": {
                "BoundingBox": {
                    "Left": 0.08,
                    "Top": 0.38,
                    "Width": 0.12,
                    "Height": 0.03
                }
            }
        },

        {
            "BlockType": "WORD",
            "Text": "SINK",
            "Geometry": {
                "BoundingBox": {
                    "Left": 0.47,
                    "Top": 0.28,
                    "Width": 0.05,
                    "Height": 0.02
                }
            }
        },

        {
            "BlockType": "WORD",
            "Text": "STOVE / COOKTOP",
            "Geometry": {
                "BoundingBox": {
                    "Left": 0.72,
                    "Top": 0.35,
                    "Width": 0.14,
                    "Height": 0.05
                }
            }
        },

        {
            "BlockType": "WORD",
            "Text": "OVEN",
            "Geometry": {
                "BoundingBox": {
                    "Left": 0.76,
                    "Top": 0.52,
                    "Width": 0.06,
                    "Height": 0.03
                }
            }
        },

        {
            "BlockType": "WORD",
            "Text": "DINING TABLE",
            "Geometry": {
                "BoundingBox": {
                    "Left": 0.43,
                    "Top": 0.88,
                    "Width": 0.16,
                    "Height": 0.03
                }
            }
        }
    ]
}

textract_output

In [ ]:
rows = []

for block in textract_output["Blocks"]:
    bb = block["Geometry"]["BoundingBox"]

    rows.append({
        "text": block["Text"],
        "left_px": int(bb["Left"] * width),
        "top_px": int(bb["Top"] * height),
        "width_px": int(bb["Width"] * width),
        "height_px": int(bb["Height"] * height)
    })

tex_df = pd.DataFrame(rows)
tex_df

In [ ]:
comparison = pd.DataFrame({
    "Capability": [
        "Returns object names",
        "Returns approximate coordinates",
        "Reads labels accurately",
        "Understands kitchen semantics",
        "Detects exact text regions",
        "Good for redaction",
        "Good for reasoning"
    ],
    
    "GPT-4o-mini": [
        "Yes",
        "Approximate center points",
        "Usually yes",
        "Excellent",
        "Weak",
        "No",
        "Excellent"
    ],
    
    "AWS Textract": [
        "Only text labels",
        "Bounding boxes only",
        "Excellent",
        "Weak",
        "Excellent",
        "Excellent",
        "Weak"
    ]
})

comparison

In [ ]:
llm_example = pd.DataFrame({
    "Object": [
        "Sink",
        "Refrigerator",
        "Dining Table"
    ],
    "LLM Predicted Center (x,y)": [
        "(640, 260)",
        "(150, 430)",
        "(640, 860)"
    ],
    "Textract Output": [
        "Bounding box around label 'SINK'",
        "Bounding box around text 'REFRIGERATOR'",
        "Bounding box around text 'DINING TABLE'"
    ]
})

llm_example

In [ ]:
## Q1. Do predicted coordinates align perfectly with the image?

No. GPT-4o-mini coordinates are approximate semantic estimates of object centers, not pixel-perfect detections.

Textract returns deterministic bounding boxes for visible text labels, which are more precise for OCR regions.

---

## Q2. Why do LLMs struggle with exact pixel-level geometry?

Multimodal LLMs convert images into compressed latent tokens optimized for meaning, relationships, and scene understanding rather than exact pixel arithmetic.

Thus they know where objects roughly are, but not exact CAD-grade coordinates.

---

## Q3. Why is GPT-4o not suitable for precise redaction tasks?

Redaction requires guaranteed pixel-level coverage of sensitive text or objects.

Approximate coordinates may:
- miss edge pixels
- over-redact nearby content
- vary across runs

Hence deterministic OCR box systems are safer.

---

## Q4. Why is Textract not suitable for semantic understanding?

Textract can detect text labels like "SINK" or "OVEN", but it does not deeply reason that:

- sink is near dishwasher
- stove is cooking zone
- island is central prep area
- kitchen workflow layout is efficient

LLMs understand these relationships.

In [ ]:
## Best Real-World Hybrid Pipeline

Use AWS Textract for:

- OCR text extraction
- precise bounding boxes
- forms / tables
- redaction coordinates

Use GPT-4o-mini / GPT-4o for:

- semantic reasoning
- layout explanation
- natural language Q&A
- spatial summaries

Combine both for enterprise-grade document intelligence.

In [ ]:
## Conclusion

Multimodal LLMs are excellent at understanding layouts conceptually, but not ideal for exact pixel geometry. Traditional structured OCR systems remain superior for deterministic coordinate tasks such as redaction and extraction zones.

The best enterprise architecture is hybrid:
geometry from OCR systems, reasoning from LLMs.